In [7]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import joblib
from library import * 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
def create_binary_datasets(dataset_path: str):
    dataset_info = {}
    dataset = pd.read_csv(f"./{dataset_path}.csv")
    # set types
    ses = get_ADRs(dataset)
    for se in ses:
        dataset[se] = dataset[se].astype(int)

    cpis = get_cpis(dataset)
    for cpi in cpis:
        dataset[cpi] = dataset[cpi].astype(int)

    fs = get_fs(dataset)
    for f in fs:
        dataset[f] = dataset[f].astype(int)

    dataset = dataset.drop(columns=["CID"])

    # Create single-label binary datasets
    ses = get_ADRs(dataset)
    for se in ses:
        print("Preparing dataset for ", se)
        dataset_with_single_label = keep_single_se(dataset, se)
        dataset_with_single_label[se] = dataset_with_single_label[se].astype(int)
        dataset_with_single_label = remove_inconsistent_duplicates(
            dataset_with_single_label, cpis + fs, [se]
        )
        dataset_info[se] = print_dataset_info(dataset_with_single_label, [se])
        dataset_with_single_label.to_csv(
            f"./binary_datasets/{dataset_path}_{se}.csv", index=False
        )
        print("\n\n\n")
    return dataset_info


def train(dataset_path: str, se: str, results: dict, scoring: dict):
    print("Training dataset:", dataset_path)
    print("Label:", se)
    dataset = pd.read_csv(f"datasets/binary_datasets/{dataset_path}_{se}.csv")
    dataset[se] = dataset[se].astype(int)

    rf_model, cv_scores, importances = train_n_test(
        "RF",
        dataset,
        None,
        se,
        True,
        scoring,
    )
    print(cv_scores)
    if dataset_path not in results:
        results[dataset_path] = {}

    if se not in results[dataset_path]:
        results[dataset_path][se] = {}
    results[dataset_path][se] = cv_scores

    joblib.dump(
        rf_model, "./binary_datasets/models/" + dataset_path + "_" + se + ".pkl"
    )
    print("Model saved successfully")


def get_br_results_with_macro(df: pd.DataFrame, dataset_paths: list, ses: list):
    column_groups = {}
    for dataset_path in dataset_paths:
        group = []
        for se in ses:
            dataset_name = dataset_path + "_" + se
            group.append(dataset_name)
        metric_name = dataset_path
        column_groups[metric_name] = group

    for new_col, cols in reversed(column_groups.items()):
        df.insert(0, new_col, df[cols].mean(axis=1))

    df = df[list(column_groups.keys())]
    return df

In [9]:
# Binary relevance - create binary datasets
dataset_paths = ["cpi_dataset", "fingerprint_dataset", "cpi_fingerprint_dataset"]
dataset_info = {}
for dataset_path in dataset_paths:
    dataset_info[dataset_path] = create_binary_datasets(dataset_path)

Preparing dataset for  se_C0011603
total ses: 6
dropping: 5

❌ Deleted Rows (non-uniform in check_cols): 2
Number of instances: 1377
Number of CPI features: 1610
Number of fingerprint features: 0
Number of features: 1610
Number of ADRs: 1




Preparing dataset for  se_C0012833
total ses: 6
dropping: 5

❌ Deleted Rows (non-uniform in check_cols): 6
Number of instances: 1375
Number of CPI features: 1610
Number of fingerprint features: 0
Number of features: 1610
Number of ADRs: 1




Preparing dataset for  se_C0015230
total ses: 6
dropping: 5

❌ Deleted Rows (non-uniform in check_cols): 2
Number of instances: 1377
Number of CPI features: 1610
Number of fingerprint features: 0
Number of features: 1610
Number of ADRs: 1




Preparing dataset for  se_C0018681
total ses: 6
dropping: 5

❌ Deleted Rows (non-uniform in check_cols): 4
Number of instances: 1376
Number of CPI features: 1610
Number of fingerprint features: 0
Number of features: 1610
Number of ADRs: 1




Preparing dataset for  se_C0

In [10]:
new_dict = {}
for path in dataset_info:
    # print(path)
    for se in dataset_info[path]:
        dataset_name = f"{path}_{se}"
        new_dict[dataset_name] = dataset_info[path][se]
    # print(path, dataset_info[path])
stats = pd.DataFrame.from_dict(new_dict, orient="index")
print(stats.to_string())

                                     Number of instances  Number of CPI features  Number of fingerprint features  Number of features  Number of ADRs
cpi_dataset_se_C0011603                             1377                    1610                               0                1610               1
cpi_dataset_se_C0012833                             1375                    1610                               0                1610               1
cpi_dataset_se_C0015230                             1377                    1610                               0                1610               1
cpi_dataset_se_C0018681                             1376                    1610                               0                1610               1
cpi_dataset_se_C0027497                             1377                    1610                               0                1610               1
cpi_dataset_se_C0042963                             1376                    1610                          

In [11]:
# Training
dataset_paths = ["cpi_dataset", "fingerprint_dataset", "cpi_fingerprint_dataset"]
ses = [
    "se_C0011603",
    "se_C0012833",
    "se_C0015230",
    "se_C0018681",
    "se_C0027497",
    "se_C0042963",
]
scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}
results = {}
for dataset_path in dataset_paths:
    for se in ses:
        train(dataset_path, se, results, scoring)
# ~12m

Training dataset: cpi_dataset
Label: se_C0011603
Number of instances: 1377
Number of CPI features: 1610
Number of fingerprint features: 0
Number of features: 1610
Number of ADRs: 1
positive examples: 1056
negative examples: 321
feature_columns: 1610
label_column: 1
{'precision': np.float64(0.7805540162515612), 'recall': np.float64(0.9185893980233603), 'f1': np.float64(0.8438603186019563), 'roc_auc': np.float64(0.6027703075226659)}
Model saved successfully
Training dataset: cpi_dataset
Label: se_C0012833
Number of instances: 1375
Number of CPI features: 1610
Number of fingerprint features: 0
Number of features: 1610
Number of ADRs: 1
positive examples: 1005
negative examples: 370
feature_columns: 1610
label_column: 1
{'precision': np.float64(0.7780058379298731), 'recall': np.float64(0.9104950495049504), 'f1': np.float64(0.8387044934467998), 'roc_auc': np.float64(0.6715953973775757)}
Model saved successfully
Training dataset: cpi_dataset
Label: se_C0015230
Number of instances: 1377
Numbe

In [12]:
records = {}
for dataset in results:
    for se in results[dataset]:
        for metric in results[dataset][se]:
            if metric not in records:
                records[metric] = {}
            records[metric][dataset + "_" + se] = results[dataset][se][metric]

results_df = pd.DataFrame.from_dict(records, orient="index")
results_df_macro = get_br_results_with_macro(results_df, dataset_paths, ses)
path = "../datasets/binary_datasets/results/"
filename = "br_results"
save_df_to_csv(results_df, path, filename)
save_df_to_csv(results_df_macro, path, filename + "_Macro")

results_df_macro.T

,precision,recall,f1,roc_auc
cpi_dataset,0.800506,0.930787,0.860535,0.636949
fingerprint_dataset,0.807203,0.931205,0.864522,0.651784
cpi_fingerprint_dataset,0.798126,0.957802,0.870308,0.664715
